In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [2]:
movies = pd.read_csv("../data/processed/movies_clean.csv")

print("Dataset shape:", movies.shape)
movies.head()

Dataset shape: (58098, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
movies["genres"] = movies["genres"].fillna("")

movies["genres_text"] = movies["genres"].str.replace(
    "|",
    " ",
    regex=False
)

movies[["title", "genres", "genres_text"]].head()

,title,genres,genres_text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


In [4]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["genres_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (58098, 23)


In [ ]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies["genres_text"]
)

In [8]:
movie_indices = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()

movie_indices.head()

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [9]:
def recommend_movies(movie_title, num_recommendations=10):

    if movie_title not in movie_indices:
        return f"Movie '{movie_title}' not found."

    movie_index = movie_indices[movie_title]

    # Calculate similarity only for the selected movie
    similarity_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    # Get indices of highest similarity
    similar_indices = similarity_scores.argsort()[
        -(num_recommendations + 1):
    ][::-1]

    # Remove the selected movie itself
    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ][:num_recommendations]

    recommendations = movies.iloc[
        similar_indices
    ][["title", "genres"]].copy()

    recommendations["similarity_score"] = [
        round(similarity_scores[i], 3)
        for i in similar_indices
    ]

    return recommendations

In [10]:
movies["title"].head(20).tolist()

['Toy Story (1995)',
 'Jumanji (1995)',
 'Grumpier Old Men (1995)',
 'Waiting to Exhale (1995)',
 'Father of the Bride Part II (1995)',
 'Heat (1995)',
 'Sabrina (1995)',
 'Tom and Huck (1995)',
 'Sudden Death (1995)',
 'GoldenEye (1995)',
 'American President, The (1995)',
 'Dracula: Dead and Loving It (1995)',
 'Balto (1995)',
 'Nixon (1995)',
 'Cutthroat Island (1995)',
 'Casino (1995)',
 'Sense and Sensibility (1995)',
 'Four Rooms (1995)',
 'Ace Ventura: When Nature Calls (1995)',
 'Money Train (1995)']

In [11]:
recommend_movies("Toy Story (1995)")

,title,genres,similarity_score
21576,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
24657,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy,1.0
25651,The Magic Crystal (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
25071,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
54897,Trolls Holiday (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
3028,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
50097,The Dragon Spell (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0
25073,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
3664,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
24734,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [12]:
recommend_movies(
    "Toy Story (1995)",
    10
)

,title,genres,similarity_score
21576,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
24657,Aladdin (1992),Adventure|Animation|Children|Comedy|Fantasy,1.0
25651,The Magic Crystal (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
25071,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
54897,Trolls Holiday (2017),Adventure|Animation|Children|Comedy|Fantasy,1.0
3028,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
50097,The Dragon Spell (2016),Adventure|Animation|Children|Comedy|Fantasy,1.0
25073,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy,1.0
3664,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
24734,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [13]:
joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

joblib.dump(
    tfidf_matrix,
    "../models/tfidf_matrix.pkl"
)

joblib.dump(
    movie_indices,
    "../models/movie_indices.pkl"
)

movies.to_pickle(
    "../models/movies.pkl"
)

print("Content-based model saved successfully!")

Content-based model saved successfully!
